In [77]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from src.feature_loader import load_features
from src.cross_validation import CrossValidator
import src.util as util

Check class distribution of samples in data splits

In [78]:
LABEL_PATH = "data/participants.tsv"
AD_FTD_CN = {"A": 0, "F": 1, "C": 2}
AD_CN = {"A": 1, "C": 0}
FTD_CN = {"F": 1, "C": 0}
AD_FTD = {"A": 1, "F": 0}
band = "alpha"

In [79]:
# Load data for AD vs CN classification
print("Loading features and labels...")
features_AD, labels_AD, subjects_AD = load_features(
    label_map=AD_CN, band=band
)
print("Data loaded.")
print("Classification task:", AD_CN)

Loading features and labels...
Min windows per subject: 303
Max windows per subject: 1278
Data loaded.
Classification task: {'A': 1, 'C': 0}


In [94]:
# Check class distribution overall
overall_AD = pd.DataFrame({"label": labels_AD, "subject": subjects_AD})
print("Overall class distribution (AD vs CN):")
print(overall_AD["label"].value_counts())
print()
print("Overall class distribution (AD vs CN) ratios:")
print(overall_AD["label"].value_counts(normalize=True))

Overall class distribution (AD vs CN):
label
1    29009
0    24011
Name: count, dtype: int64

Overall class distribution (AD vs CN) ratios:
label
1    0.547133
0    0.452867
Name: proportion, dtype: float64


In [95]:
# Check class distribution in data splits

results_AD = []

# Set up outer cross-validation (LOSO)
outer_cv = CrossValidator(
    features=features_AD, labels=labels_AD, subjects=subjects_AD, 
    cv_strategy="loso", n_splits=None, test_size=None, shuffle=False, 
    random_state=123)

for fold, train_val_idx, test_idx in outer_cv.cv_loop():

    # Outer train/val class distribution
    outer_counts = pd.Series(labels_AD[train_val_idx]).value_counts()
    outer_ratios = pd.Series(labels_AD[train_val_idx]).value_counts(normalize=True)

    # Inner single split for train/val
    inner_split = CrossValidator(
        features=features_AD[train_val_idx],
        labels=labels_AD[train_val_idx],
        subjects=subjects_AD[train_val_idx],
        cv_strategy=None,
        n_splits=None,
        test_size=0.2,
        shuffle=True,
        random_state=123
    )
    _, train_idx, val_idx = next(inner_split.cv_loop())

    # Map back to original indices
    train_idx = train_val_idx[train_idx]
    val_idx = train_val_idx[val_idx]

    # Inner train class distribution
    inner_train_counts = pd.Series(labels_AD[train_idx]).value_counts()
    inner_train_ratios = pd.Series(labels_AD[train_idx]).value_counts(normalize=True)

    results_AD.append({
        "fold": fold,
        "test_subject": subjects_AD[test_idx][0],
        "outer_train_val_ratio_A": outer_ratios[1],
        "outer_train_val_ratio_C": outer_ratios[0],
        "inner_train_ratio_A": inner_train_ratios[1],
        "inner_train_ratio_C": inner_train_ratios[0]
    })


CV Strategy: Leave One Subject Out (LOSO)
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test spli

In [96]:
# Create DataFrame of results and calculate imbalance ratios
df_AD = pd.DataFrame(results_AD)

df_AD["outer_imbalance"] = (df_AD["outer_train_val_ratio_A"] - 0.5).abs()
df_AD["inner_imbalance"] = (df_AD["inner_train_ratio_A"] - 0.5).abs()

outer_cv_results = df_AD[["fold", "test_subject", "outer_train_val_ratio_A", "outer_train_val_ratio_C", "outer_imbalance"]]
inner_cv_results = df_AD[["fold", "test_subject", "inner_train_ratio_A", "inner_train_ratio_C", "inner_imbalance"]]

In [97]:
# Display outer CV results
print("Outer CV train/val class ratios (AD vs CN):")
outer_cv_results.sort_values(by="outer_imbalance", ascending=False)

Outer CV train/val class ratios (AD vs CN):


,fold,test_subject,outer_train_val_ratio_A,outer_train_val_ratio_C,outer_imbalance
47,47,sub-048,0.557501,0.442499,0.057501
39,39,sub-040,0.557244,0.442756,0.057244
41,41,sub-042,0.557222,0.442778,0.057222
61,61,sub-062,0.556528,0.443472,0.056528
37,37,sub-038,0.556453,0.443547,0.056453
...,...,...,...,...,...
13,13,sub-014,0.539039,0.460961,0.039039
33,33,sub-034,0.538729,0.461271,0.038729
15,15,sub-016,0.538702,0.461298,0.038702
30,30,sub-031,0.537120,0.462880,0.037120


In [87]:
# Display inner splits results
print("Inner split train class ratios (AD vs CN):")
inner_cv_results.sort_values(by="inner_imbalance", ascending=False)

Inner split train class ratios (AD vs CN):


,fold,test_subject,inner_train_ratio_A,inner_train_ratio_C,inner_imbalance
57,57,sub-058,0.581946,0.418054,0.081946
61,61,sub-062,0.580996,0.419004,0.080996
20,20,sub-021,0.580865,0.419135,0.080865
33,33,sub-034,0.580013,0.419987,0.080013
25,25,sub-026,0.578926,0.421074,0.078926
...,...,...,...,...,...
2,2,sub-003,0.529077,0.470923,0.029077
8,8,sub-009,0.528224,0.471776,0.028224
5,5,sub-006,0.527732,0.472268,0.027732
4,4,sub-005,0.526174,0.473826,0.026174


In [88]:
# Load data for FTD vs CN classification
print("Loading features and labels...")
features_FTD, labels_FTD, subjects_FTD = load_features(
    label_map=FTD_CN, band=band
)
print("Data loaded.")
print("Classification task:", FTD_CN)

Loading features and labels...
Min windows per subject: 476
Max windows per subject: 1011
Data loaded.
Classification task: {'F': 1, 'C': 0}


In [89]:
# Check classification label distribution overall
overall_FTD = pd.DataFrame({"label": labels_FTD, "subject": subjects_FTD})
print("Overall class distribution (FTD vs CN):")
print(overall_FTD["label"].value_counts())
print()
print("Overall class distribution (FTD vs CN) ratios:")
print(overall_FTD["label"].value_counts(normalize=True))

Overall class distribution (FTD vs CN):
label
0    24011
1    16510
Name: count, dtype: int64

Overall class distribution (FTD vs CN) ratios:
label
0    0.592557
1    0.407443
Name: proportion, dtype: float64


In [90]:
# Check class distribution in data splits

results_FTD = []

# Set up outer cross-validation (LOSO)
outer_cv = CrossValidator(
    features=features_FTD, labels=labels_FTD, subjects=subjects_FTD, 
    cv_strategy="loso", n_splits=None, test_size=None, shuffle=False, 
    random_state=123)

for fold, train_val_idx, test_idx in outer_cv.cv_loop():

    # Outer train/val class distribution
    outer_counts = pd.Series(labels_FTD[train_val_idx]).value_counts()
    outer_ratios = pd.Series(labels_FTD[train_val_idx]).value_counts(normalize=True)

    # Inner single split for train/val
    inner_split = CrossValidator(
        features=features_FTD[train_val_idx],
        labels=labels_FTD[train_val_idx],
        subjects=subjects_FTD[train_val_idx],
        cv_strategy=None,
        n_splits=None,
        test_size=0.2,
        shuffle=True,
        random_state=123
    )
    _, train_idx, val_idx = next(inner_split.cv_loop())

    # Map back to original indices
    train_idx = train_val_idx[train_idx]
    val_idx = train_val_idx[val_idx]

    # Inner train class distribution
    inner_train_counts = pd.Series(labels_FTD[train_idx]).value_counts()
    inner_train_ratios = pd.Series(labels_FTD[train_idx]).value_counts(normalize=True)

    results_FTD.append({
        "fold": fold,
    "test_subject": subjects_FTD[test_idx][0],
        "outer_train_val_ratio_F": outer_ratios[1],
        "outer_train_val_ratio_C": outer_ratios[0],
        "inner_train_ratio_F": inner_train_ratios[1],
        "inner_train_ratio_C": inner_train_ratios[0]
    })


CV Strategy: Leave One Subject Out (LOSO)
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test split.
Using single train-test spli

In [91]:
# Create DataFrame of results and calculate imbalance ratios
df_FTD = pd.DataFrame(results_FTD)

df_FTD["outer_imbalance"] = (df_FTD["outer_train_val_ratio_F"] - 0.5).abs()
df_FTD["inner_imbalance"] = (df_FTD["inner_train_ratio_F"] - 0.5).abs()

outer_cv_results = df_FTD[["fold", "test_subject", "outer_train_val_ratio_F", "outer_train_val_ratio_C", "outer_imbalance"]]
inner_cv_results = df_FTD[["fold", "test_subject", "inner_train_ratio_F", "inner_train_ratio_C", "inner_imbalance"]]

In [92]:
# Display outer CV results
print("Outer CV train/val class ratios (FTD vs CN):")
outer_cv_results.sort_values(by="outer_imbalance", ascending=False)

Outer CV train/val class ratios (FTD vs CN):


,fold,test_subject,outer_train_val_ratio_F,outer_train_val_ratio_C,outer_imbalance
37,37,sub-074,0.392280,0.607720,0.107720
43,43,sub-080,0.393799,0.606201,0.106201
46,46,sub-083,0.393830,0.606170,0.106170
41,41,sub-078,0.394487,0.605513,0.105513
36,36,sub-073,0.394732,0.605268,0.105268
44,44,sub-081,0.395189,0.604811,0.104811
39,39,sub-076,0.395280,0.604720,0.104720
42,42,sub-079,0.395311,0.604689,0.104689
51,51,sub-088,0.395798,0.604202,0.104202
45,45,sub-082,0.395904,0.604096,0.104096


In [93]:
# Display inner splits results
print("Inner split train class ratios (FTD vs CN):")
inner_cv_results.sort_values(by="inner_imbalance", ascending=False)

Inner split train class ratios (FTD vs CN):


,fold,test_subject,inner_train_ratio_F,inner_train_ratio_C,inner_imbalance
51,51,sub-088,0.392837,0.607163,0.107163
43,43,sub-080,0.396738,0.603262,0.103262
27,27,sub-064,0.401501,0.598499,0.098499
38,38,sub-075,0.411230,0.588770,0.088770
24,24,sub-061,0.411382,0.588618,0.088618
19,19,sub-056,0.411420,0.588580,0.088580
46,46,sub-083,0.411431,0.588569,0.088569
20,20,sub-057,0.411482,0.588518,0.088518
22,22,sub-059,0.411870,0.588130,0.088130
40,40,sub-077,0.412127,0.587873,0.087873
